# 🎯 Object Detection — training on Colab

This notebook runs **the same** training as `scripts/train.py`, on Colab's free
GPU. What takes ~45 s/epoch on a Mac drops to a few seconds here.

**How to use it:** pick `Runtime → Change runtime type → T4 GPU`, then run the
cells in order.

At the end, download the trained `best.pt` and drop it into the project's
`models/` folder — the app picks it up automatically as *"Custom: ..."*.


## 1. Check the GPU


In [ ]:
!nvidia-smi

## 2. Setup

`lap` is needed for ByteTrack. We install it even though this notebook does not
track anything, to keep the dependencies identical to the project.


In [ ]:
!pip install -q ultralytics 'lap>=0.5.12'

import torch
import ultralytics

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
print("ultralytics:", ultralytics.__version__)

## 3. Settings

To train on your own data, point `DATA` at your own `data.yaml`.


In [ ]:
DATA = "african-wildlife.yaml"  # a built-in ultralytics set, or your own data.yaml
MODEL = "yolov8n.pt"  # starting weights
EPOCHS = 60  # we can afford plenty on a GPU
IMGSZ = 640
BATCH = 32
NAME = DATA.replace(".yaml", "")

## 4. Training

`patience=15`: stop early if 15 epochs pass without improvement, rather than
burning GPU time for nothing.


In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
results = model.train(
    data=DATA,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=15,
    name=NAME,
    plots=True,
)
print("output:", results.save_dir)

## 5. Validation metrics

Compare this table with the one in the README.


In [ ]:
metrics = model.val(data=DATA)

print(f"mAP50    : {metrics.box.map50:.3f}")
print(f"mAP50-95 : {metrics.box.map:.3f}")
print(f"precision: {metrics.box.mp:.3f}")
print(f"recall   : {metrics.box.mr:.3f}")
print()
for i, c in enumerate(metrics.box.ap_class_index):
    print(f"{model.names[int(c)]:12} mAP50={metrics.box.ap50[i]:.3f}")

## 6. Training plots


In [ ]:
from pathlib import Path

from IPython.display import Image, display

run = Path(results.save_dir)
for plot in ["results.png", "confusion_matrix_normalized.png", "BoxPR_curve.png"]:
    path = run / plot
    if path.exists():
        print(plot)
        display(Image(filename=str(path), width=800))

## 7. Download the model

Put the downloaded file into the project's `models/` folder as `african-wildlife.pt`.


In [ ]:
from pathlib import Path

from google.colab import files

best = Path(results.save_dir) / "weights" / "best.pt"
target = Path(f"/content/{NAME}.pt")
target.write_bytes(best.read_bytes())
print("size:", round(target.stat().st_size / 1e6, 1), "MB")
files.download(str(target))